# Daten-Bereinigung
Ziel von diesem Notebook ist die Bereinigung des Rohdatensatzes Swissvotes. Hierfür wird die bestehenden CSV-Datei geladen, entsprechende Spalten selektiert und die für das Projekt passenden Datentypen zugewiesen. Es werden keine Datensätze gefiltert oder gelöscht.
## Import Pakete

In [ ]:
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2
from visualisierungen import *

## Datenset laden

In [ ]:
data = pd.read_csv('../data/raw/DATASET CSV 11-02-2026.csv', sep=';')
display(data.head())
display(data.tail())

In [ ]:
print('svp', data['p-svp'].value_counts())
print('mitte', data['p-mitte'].value_counts())
print('sps', data['p-sps'].value_counts())

### Struktur der Daten

In [ ]:
print(f'Der Datensatz hat {data.shape[0]} Zeilen und {data.shape[1]} Spalten.') #Ausprägungen anzeigen
display(pd.DataFrame(data.columns, columns=['Spalten'])) #head, tail Merkmalnamen

df_types = data.dtypes.value_counts().reset_index() #Ausprägung pro Datentyp
df_types.columns = ['Datentyp', 'Anzahl']
display(df_types)

display(pd.DataFrame(data.isna().sum(),columns=['na'])) #na/ null in head, tail

### Hauptthemen analysieren

In [ ]:
unique_titel = data['titel_kurz_d'].unique()
print(f'Der Datensatz hat {len(unique_titel)} verschiedene Titel.') #Ausprägungen anzeigen
display(pd.DataFrame(unique_titel, columns=['titel_kurz_d']))

## Wrangling und Datacleaning

### Datatypes

#### Existierende Typen

In [ ]:
data_filtered = data.loc[:,['datum','legisjahr','d1e1', 'd1e2', 'd1e3', 'br-pos', 'annahme',
                'volkja-proz', 'berecht', 'stimmen', 'gultig',
                'leer', 'rechtsform', 'zh-japroz', 'be-japroz', 'lu-japroz', 'ur-japroz', 'sz-japroz',
                'ow-japroz', 'nw-japroz', 'gl-japroz', 'zg-japroz', 'fr-japroz',
                'so-japroz', 'bs-japroz', 'bl-japroz', 'sh-japroz', 'ar-japroz',
                'ai-japroz', 'sg-japroz', 'gr-japroz', 'ag-japroz', 'tg-japroz',
                'ti-japroz', 'vd-japroz', 'vs-japroz', 'ne-japroz', 'ge-japroz',
                'ju-japroz',]]
data_filtered.dtypes

#### Formatierung Typen

In [ ]:
data['datum'] = pd.to_datetime(data['datum'], format='%d.%m.%Y') # Formatierung Datum von DD.MM.YYYY zu YYYY-MM-DD
data['jahr'] = data['datum'].dt.year # Formatierung Jahr zu Format YYYY

# Formatierung Werte zu float
numeric_cols = ['d1e1', 'd1e2', 'd1e3', 'br-pos', 'annahme',
                'volkja-proz', 'berecht', 'stimmen', 'gultig',
                'leer', 'rechtsform', 'zh-japroz', 'be-japroz', 'lu-japroz', 'ur-japroz', 'sz-japroz',
                'ow-japroz', 'nw-japroz', 'gl-japroz', 'zg-japroz', 'fr-japroz',
                'so-japroz', 'bs-japroz', 'bl-japroz', 'sh-japroz', 'ar-japroz',
                'ai-japroz', 'sg-japroz', 'gr-japroz', 'ag-japroz', 'tg-japroz',
                'ti-japroz', 'vd-japroz', 'vs-japroz', 'ne-japroz', 'ge-japroz',
                'ju-japroz',]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce') # numeric entspricht float, was nicht umgewandelt werden kann, wird zu NaN.

data['bet'] = data['bet'].replace(r'^\s*\.\s*$', np.nan, regex=True)
data['beteiligung'] = pd.to_numeric(data['bet'], errors='coerce')

parteien = [
    'p-fdp', 'p-sps', 'p-svp', 'p-mitte', 'p-evp', 'p-gps', 'p-glp',
    'p-ucsp', 'p-pda', 'p-sd', 'p-edu', 'p-fps', 'p-lega', 'p-kvp',
    'p-mcg', 'p-cvp', 'p-bdp', 'p-lps', 'p-ldu', 'p-poch', 'p-rep',
    'p-eco', 'p-sgv', 'p-sbv', 'p-sgb', 'p-travs', 'p-sav', 'p-vsa',
    'p-vpod', 'p-voev', 'p-tcs', 'p-vcs', 'p-acs', 'p-sbk', 'p-ssv',
    'p-gem', 'p-kdk', 'p-kkjpd', 'p-gdk', 'p-ldk', 'p-vdk', 'p-sodk',
    'p-endk', 'p-fdk', 'p-edk', 'p-bpuk'
]

for col in parteien:
    if col in data.columns:
        data[col] = data[col].replace(r'^\s*\.\s*$', np.nan, regex=True)
        data[col] = data[col].replace(9999, np.nan)
        data[col] = pd.to_numeric(data[col], errors='coerce')

data.replace([9999,999], np.nan, inplace=True)
print(data.head())

In [ ]:
# TODO: - We have to Clean up categories.
# TODO: - We need to merge parties cvp --> became mitte, what are we gonna do with BDP?

## Variabeln/Zeilen löschen

In [ ]:
data.dropna(subset=['volkja-proz'], inplace=True)

## Variabeln erstellen

In [ ]:
data['jahrzehnt'] = (data['jahr'] // 10 * 10).astype(str)

## Mapping

### Thematische Gruppen der Abstimmungen
Fokus auf Einteilung durch Swissvote (in Abstimmung mit BFS), da es sich um eine inhaltliche Überarbeitung der ursprünglichen BFS Position geht. Deshalb werden d2 und d3 nicht berücksichtigt.

In [ ]:
data.head()

In [ ]:
# Hauptgruppe (d1e1)
def assign_hauptgruppe(row):
    d1 = row['d1e1']
    mapping = {
        1: 'Staatsordnung',
        2: 'Aussenpolitik',
        3: 'Sicherheitspolitik',
        4: 'Wirtschaft',
        5: 'Landwirtschaft',
        6: 'Öffentliche Finanzen',
        7: 'Energie',
        8: 'Verkehr & Infrastruktur',
        9: 'Umwelt & Lebensraum',
        10: 'Sozial- und Gesellschaftspolitik',
        11: 'Bildung & Forschung',
        12: 'Kultur, Religion & Medien',
    }
    return mapping.get(d1, 'Andere')

def assign_untergruppen(row):
    d2 = row['d1e2']

    mapping = {
        1.1: 'Nationale Identität',
        1.2: 'Politisches System',
        1.3: 'Institutionen',
        1.4: 'Volksrechte',
        1.5: 'Föderalismus',
        1.6: 'Rechtsordnung',
        2.1: 'Aussenpolitische Grundhaltung',
        2.2: 'Europapolitik',
        2.3: 'Internationale Organisationen',
        2.4: 'Entwicklungszusammenarbeit',
        2.5: 'Staatsverträge',
        2.6: 'Aussenwirtschaftspolitik',
        2.7: 'Diplomatie',
        2.8: 'Auslandschweizer:innen',
        3.1: 'Öffentliche Sicherheit',
        3.2: 'Armee',
        3.3: 'Landesversorgung',
        4.1: 'Wirtschaftspolitik',
        4.2: 'Arbeit und Beschäftigung',
        4.3: 'Finanzwesen',
        4.4: 'Freizeit und Tourismus',
        5.1: 'Agrarpolitik',
        5.2: 'Tierische Produktion',
        5.3: 'Pflanzliche Produktion',
        5.4: 'Forstwirtschaft',
        5.5: 'Fischerei, Jagd, Haustiere',
        6.1: 'Steuerwesen',
        6.2: 'Finanzordnung',
        6.3: 'Öffentliche Ausgaben',
        6.4: 'Spar- und Sanierungsmassnahmen',
        7.1: 'Energiepolitik',
        7.2: 'Kernenergie',
        7.3: 'Wasserkraft',
        7.4: 'Alternativenergien',
        7.5: 'Erdöl, Gas',
        8.1: 'Verkehrspolitik',
        8.2: 'Strassenverkehr',
        8.3: 'Schienenverkehr',
        8.4: 'Luftverkehr',
        8.5: 'Schifffahrt',
        8.6: 'Post',
        8.7: 'Telekommunikation',
        9.1: 'Boden',
        9.2: 'Wohnen',
        9.3: 'Umwelt',
        10.1: 'Gesundheit',
        10.2: 'Sozialversicherungen',
        10.3: 'Gesellschaftsfragen',
        11.1: 'Bildungspolitik',
        11.2: 'Schulen',
        11.3: 'Hochschulen',
        11.4: 'Forschung',
        11.5: 'Berufsbildung',
        12.1: 'Kulturpolitik',
        12.2: 'Sprachpolitik',
        12.3: 'Religion, Kirchen',
        12.4: 'Sport',
        12.5: 'Medien und Kommunikation',
    }
    return mapping.get(d2, 'Andere')

# Kleinstgruppe (d1e3)
def assign_kleinstgruppe(row):
    d3 = row['d1e3']
    mapping = {
        1.21: 'Bundesverfassung',
        1.22: 'Verfassungsgebungsverfahren',
        1.23: 'Gesetzgebungsverfahren',
        1.24: 'Wahlsystem',
        1.31: 'Regierung, Verwaltung',
        1.32: 'Parlament',
        1.33: 'Gerichte',
        1.34: 'Nationalbank',
        1.41: 'Initiative',
        1.42: 'Referendum',
        1.43: 'Stimmrecht',
        1.51: 'Territorialfragen',
        1.52: 'Beziehungen Bund–Kantone',
        1.53: 'Aufgabenteilung',
        1.61: 'Internationales Recht',
        1.62: 'Grundrechte',
        1.63: 'Bürgerrecht',
        1.64: 'Privatrecht',
        1.65: 'Strafrecht',
        1.66: 'Datenschutz',
        2.11: 'Neutralität',
        2.12: 'Unabhängigkeit',
        2.13: 'Gute Dienste',
        2.21: 'EFTA',
        2.22: 'EU',
        2.23: 'EWR',
        2.24: 'Andere europäische Organisationen',
        2.31: 'UNO',
        2.32: 'Andere internationale Organisationen',
        2.61: 'Exportförderung',
        2.62: 'Zollwesen',
        3.11: 'Bevölkerungsschutz',
        3.12: 'Staatsschutz',
        3.13: 'Polizei',
        3.21: 'Armee (allgemein)',
        3.22: 'Militärorganisation',
        3.23: 'Rüstung',
        3.24: 'Militäranlagen',
        3.25: 'Dienstverweigerung, Zivildienst',
        3.26: 'Armeeabschaffung',
        3.27: 'Militärische Ausbildung',
        3.28: 'Internationale Einsätze',
        4.11: 'Konjunkturpolitik',
        4.12: 'Wettbewerbspolitik',
        4.13: 'Strukturpolitik',
        4.14: 'Preispolitik',
        4.15: 'Konsumentenschutz',
        4.16: 'Gesellschaftsrecht',
        4.21: 'Arbeitsbedingungen',
        4.22: 'Arbeitszeit',
        4.23: 'Sozialpartnerschaft',
        4.24: 'Beschäftigungspolitik',
        4.31: 'Geld- und Währungspolitik',
        4.32: 'Banken, Börsen, Versicherungen',
        4.41: 'Fremdenverkehr',
        4.42: 'Hotellerie und Gastgewerbe',
        4.43: 'Geldspiele',
        6.11: 'Steuerpolitik',
        6.12: 'Steuersystem',
        6.13: 'Direkte Steuern',
        6.14: 'Indirekte Steuern',
        8.11: 'Agglomerationsverkehr',
        8.12: 'Transitverkehr',
        8.21: 'Strassenbau',
        8.22: 'Schwerverkehr',
        8.31: 'Güterverkehr',
        8.32: 'Personenverkehr',
        9.11: 'Raumplanung',
        9.12: 'Bodenrecht',
        9.21: 'Mietwesen',
        9.22: 'Wohnungsbau, Wohneigentum',
        9.31: 'Umweltpolitik',
        9.32: 'Lärmschutz',
        9.33: 'Luftreinhaltung',
        9.34: 'Gewässerschutz',
        9.35: 'Bodenschutz',
        9.36: 'Abfälle',
        9.37: 'Natur- und Heimatschutz',
        9.38: 'Tierschutz',
        10.11: 'Gesundheitspolitik',
        10.12: 'Medizinforschung und -technik',
        10.13: 'Medikamente',
        10.14: 'Suchtmittel',
        10.15: 'Fortpflanzungsmedizin',
        10.21: 'AHV',
        10.22: 'Invalidenversicherung',
        10.23: 'Berufliche Vorsorge',
        10.24: 'Kranken- und Unfallversicherung',
        10.25: 'Mutterschaftsversicherung',
        10.26: 'Arbeitslosenversicherung',
        10.27: 'Erwerbsersatzordnung',
        10.28: 'Fürsorge',
        10.31: 'Migrations- und Integrationspolitik',
        10.32: 'Asylpolitik',
        10.33: 'Frauen und Gleichstellungspolitik',
        10.34: 'Familienpolitik',
        10.35: 'Kinder- und Jugendpolitik',
        10.36: 'Alterspolitik',
        10.37: 'Menschen mit Behinderungen',
        10.38: 'LGBTQIA+',
        11.41: 'Gentechnologie',
        11.42: 'Tierversuche',
        12.51: 'Medienpolitik',
        12.52: 'Presse',
        12.53: 'Radio, Fernsehen, Elektronische Medien',
        12.54: 'Medienfreiheit',
    }
    return mapping.get(d3, None)


# Anwendung
data['hauptgruppe'] = data.apply(assign_hauptgruppe, axis=1)
data['untergruppe'] = data.apply(assign_untergruppen, axis=1)
data['kleinstgruppe'] = data.apply(assign_kleinstgruppe, axis=1)


print('N hauptgruppe', data['hauptgruppe'].value_counts(dropna=False))
print('N untergruppe', data['untergruppe'].value_counts(dropna=False))
print('N kleinstgruppe', data['kleinstgruppe'].value_counts(dropna=False))

In [ ]:
data[['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3', 'hauptgruppe', 'untergruppe', 'kleinstgruppe']].sample(20, random_state=42)


### Rechtsform

In [ ]:
rechtsform_map = {
    1: 'Obligatorisches Referendum',
    2: 'Fakultatives Referendum',
    3: 'Volksinitiative',
    4: 'Direkter Gegenentwurf',
    5: 'Stichfrage'
}

data['rechtsform_name'] = data['rechtsform'].map(rechtsform_map)
print(data['rechtsform_name'])

### Position des Bundesrates

In [ ]:
position_bundesrat_map = {
    1: 'Befürwortend',
    2: 'Ablehnend',
    3: 'Keine',
    8: 'Vorzug für den Gegenentwurf (bei Stichfragen)',
    9: 'Vorzug für die Volksinitiative (bei Stichfragen)'
}

data['br-pos_name'] = data['br-pos'].map(position_bundesrat_map)
print(data['br-pos_name'])

In [ ]:
### Institutionen

#### Parteien
Fokus der Analyse auf
SVP
SP
FDP
Die Mitte
Grüne
GLP

In [ ]:
# Umbennenung Mitte berücksichtigen (2021)
print(data[['p-mitte', 'p-cvp']].value_counts())

data['p-mitte'] = data['p-cvp'].fillna(data['p-mitte'])
data.drop(columns=['p-cvp'], inplace=True)
print(data['p-mitte'].value_counts())

### Kantonsangaben
In alten Notebooks nachvollziehbar, noch nicht klar ob wir es brauchen

### Ev. Als Kontrolle die Wählendenanteile bei den letzten nationalen Wahlen

In [ ]:
x = ['w-fdp', 'w-sps', 'w-svp', 'w-mitte', 'w-evp', 'w-gps', 'w-glp', 'w-ucsp', 'w-pda', 'w-sd', 'w-edu', 'w-fps', 'w-lega', 'w-kvp', 'w-mcg', 'w-cvp', 'w-bdp', 'w-lps', 'w-ldu', 'w-poch', 'w-rep','w-ubrige']

# Neuer DF in einem separaten CSV erstellen

In [ ]:
df = data[['anr', 'datum', 'legisjahr', 'jahrzehnt', 'rechtsform_name', 'titel_kurz_d', 'anzahl','beteiligung', 'annahme', 'volkja-proz', 'berecht', 'stimmen', 'gultig', 'leer', 'hauptgruppe', 'untergruppe', 'kleinstgruppe', 'br-pos', 'br-pos_name' ,'sr-pos', 'nr-pos', 'bv-pos', 'srja', 'srnein', 'nrja', 'nrnein',  'sammeltempo','p-fdp', 'p-sps', 'p-svp', 'p-mitte', 'p-evp', 'p-gps', 'p-glp', 'p-ucsp', 'p-pda', 'p-sd', 'p-edu', 'p-fps', 'p-lega', 'p-kvp', 'p-mcg', 'p-cvp', 'p-bdp', 'p-lps', 'p-ldu', 'p-poch', 'p-rep', 'p-eco', 'p-sgv', 'p-sbv', 'p-sgb', 'p-travs', 'p-sav', 'p-vsa', 'p-vpod', 'p-voev', 'p-tcs', 'p-vcs', 'p-acs', 'p-sbk', 'p-ssv','p-gem', 'p-kdk', 'p-kkjpd', 'p-gdk', 'p-ldk', 'p-vdk', 'p-sodk', 'p-endk', 'p-fdk', 'p-edk', 'p-bpuk', 'anneepolitique', 'curiavista-de', 'swissvoteslink', 'rechtsform', 'zh-japroz', 'be-japroz', 'lu-japroz', 'ur-japroz', 'sz-japroz', 'ow-japroz', 'nw-japroz', 'gl-japroz', 'zg-japroz', 'fr-japroz', 'so-japroz', 'bs-japroz', 'bl-japroz', 'sh-japroz', 'ar-japroz', 'ai-japroz', 'sg-japroz', 'gr-japroz', 'ag-japroz', 'tg-japroz', 'ti-japroz', 'vd-japroz', 'vs-japroz', 'ne-japroz', 'ge-japroz', 'ju-japroz',]]
df.head()

In [ ]:
df.to_csv('../data/processed/swissvotes_processed.csv', index=False)